# 01_LABEL_ALL_EDI_SIGNALS

Reuse the existing Edi preprocessing and breath-detection logic, then assign every sample to one of `breath`, `apnea`, or `not_known`.

- Input root: `stored_results/00_edi_signal`
- Output root: `stored_results/01_all_duration_segments`
- Segment labels: `breath | apnea | not_known`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable
import re

import numpy as np
import pandas as pd


# =========================
# User Arguments (edit here)
# =========================
PARENT_DIR = Path("/home/jhkim/NAVA/04_LABEL_ALL_DURATION/stored_results/00_edi_signal")
ANALYSIS_FOLDER = "20260301"

TIMESTAMP_COL = "timestamp"
EDI_COL = "edi"
PEAK_COLS = ["detected_peak", "gt_peak"]
PEAK_MERGE_SEC = 0.25

APNEA_LABEL_STRATEGY = "any"  # "any" | "all" | "specific"
SPECIFIC_APNEA_LABELERS = ["apnea_label_김재호", "apnea_label_오창준"]

BASELINE_MEDIAN_SEC = 20.0
SMOOTHING_SEC = 0.4
SAMPLE_RATE_HZ = None

MIN_BREATH_SEC = 1.0
MAX_BREATH_SEC = 10.0
EDGE_EXCLUDE_SEC = 10.0

OUT_ROOT = Path("/home/jhkim/NAVA/04_LABEL_ALL_DURATION/stored_results/01_all_duration_segments")


@dataclass
class PatientRunResult:
    patient_id: str
    n_rows: int
    n_breath_segments: int
    n_apnea_segments: int
    n_not_known_segments: int
    n_total_segments: int
    status: str
    message: str = ""


In [ ]:
def to_bool_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) > 0
    text = s.astype(str).str.strip().str.lower()
    true_set = {"1", "true", "t", "yes", "y"}
    return text.isin(true_set)


def infer_sample_rate_hz(ts: np.ndarray, sample_rate_override: float | None) -> float:
    if sample_rate_override is not None:
        return float(sample_rate_override)
    dt = np.diff(ts)
    dt = dt[np.isfinite(dt)]
    dt = dt[dt > 0]
    if len(dt) == 0:
        raise ValueError("Cannot infer sample rate from timestamp.")
    med_dt = float(np.median(dt))
    return 1000.0 / med_dt if med_dt > 1.5 else 1.0 / med_dt


def rolling_median_np(arr: np.ndarray, win: int) -> np.ndarray:
    return pd.Series(arr).rolling(window=win, center=True, min_periods=1).median().to_numpy()


def rolling_mean_np(arr: np.ndarray, win: int) -> np.ndarray:
    return pd.Series(arr).rolling(window=win, center=True, min_periods=1).mean().to_numpy()


def get_patient_id_from_path(path: Path) -> str:
    m = re.search(r"patient_(\d+)\.xlsx$", path.name)
    if not m:
        return path.stem
    return m.group(1)


def build_apnea_mask(df: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    apnea_label_cols = [c for c in df.columns if c.startswith("apnea_label_")]
    if not apnea_label_cols:
        raise KeyError("No apnea_label_* columns found.")

    if APNEA_LABEL_STRATEGY == "specific":
        use_cols = [c for c in SPECIFIC_APNEA_LABELERS if c in apnea_label_cols]
        if not use_cols:
            raise KeyError("No SPECIFIC_APNEA_LABELERS columns found.")
    else:
        use_cols = apnea_label_cols

    votes = np.column_stack([to_bool_series(df[c]).to_numpy() for c in use_cols])
    if APNEA_LABEL_STRATEGY == "all":
        mask = votes.all(axis=1)
    else:
        mask = votes.any(axis=1)
    return mask.astype(bool), use_cols


def merge_close_peaks(candidate_idx: np.ndarray, time_sec: np.ndarray, y: np.ndarray) -> np.ndarray:
    if len(candidate_idx) == 0:
        return np.array([], dtype=int)

    merged: list[int] = []
    cluster = [int(candidate_idx[0])]
    for idx in candidate_idx[1:]:
        idx = int(idx)
        if (time_sec[idx] - time_sec[cluster[-1]]) <= PEAK_MERGE_SEC:
            cluster.append(idx)
        else:
            merged.append(max(cluster, key=lambda j: y[j]))
            cluster = [idx]
    merged.append(max(cluster, key=lambda j: y[j]))
    return np.array(sorted(set(merged)), dtype=int)


def build_breaths_all(
    ts: np.ndarray,
    edi_raw: np.ndarray,
    edi_detrended: np.ndarray,
    edi_smooth: np.ndarray,
    peak_idx: np.ndarray,
    analysis_window_mask: np.ndarray,
    patient_id: str,
    is_ms: bool,
) -> pd.DataFrame:
    if len(peak_idx) < 2:
        return pd.DataFrame()

    valleys: list[int] = []
    for i in range(len(peak_idx) - 1):
        left = int(peak_idx[i])
        right = int(peak_idx[i + 1])
        if right <= left:
            continue
        v_rel = np.argmin(edi_smooth[left : right + 1])
        valleys.append(left + int(v_rel))
    valleys = sorted(set(valleys))

    records = []
    breath_id = 0
    for i in range(len(valleys) - 1):
        v_start = valleys[i]
        v_end = valleys[i + 1]
        if v_end <= v_start:
            continue

        if not (edi_detrended[v_start] < 0 and edi_detrended[v_end] < 0):
            continue
        if not (analysis_window_mask[v_start] and analysis_window_mask[v_end]):
            continue

        dur = (ts[v_end] - ts[v_start]) / (1000.0 if is_ms else 1.0)
        if dur < MIN_BREATH_SEC or dur > MAX_BREATH_SEC:
            continue

        records.append(
            {
                "patient_id": patient_id,
                "breath_id": breath_id,
                "breath_label": f"breath_{breath_id:04d}",
                "v_start_idx": int(v_start),
                "v_end_idx": int(v_end),
                "v_start_t": float(ts[v_start]),
                "v_end_t": float(ts[v_end]),
                "breath_duration_sec": float(dur),
                "n_peaks": int(np.sum((peak_idx >= v_start) & (peak_idx <= v_end))),
                "edi_signal_raw": edi_raw[v_start : v_end + 1].astype(float).tolist(),
                "edi_signal_detrended": edi_detrended[v_start : v_end + 1].astype(float).tolist(),
                "edi_signal_smooth": edi_smooth[v_start : v_end + 1].astype(float).tolist(),
            }
        )
        breath_id += 1

    return pd.DataFrame(records)


def assign_sample_labels(n_rows: int, apnea_mask: np.ndarray, breaths_df: pd.DataFrame) -> tuple[np.ndarray, pd.Series]:
    labels = np.full(n_rows, "not_known", dtype=object)
    breath_id = pd.array([pd.NA] * n_rows, dtype="Int64")

    labels[apnea_mask] = "apnea"

    for row in breaths_df.itertuples(index=False):
        start_idx = int(row.v_start_idx)
        end_idx = int(row.v_end_idx)
        breath_region = ~apnea_mask[start_idx : end_idx + 1]
        labels[start_idx : end_idx + 1][breath_region] = "breath"
        for sample_idx in np.flatnonzero(breath_region) + start_idx:
            breath_id[sample_idx] = int(row.breath_id)

    return labels, pd.Series(breath_id, dtype="Int64")


def build_segment_catalog(
    patient_id: str,
    ts: np.ndarray,
    time_sec: np.ndarray,
    labels: np.ndarray,
    breath_id_per_sample: pd.Series,
    edi_raw: np.ndarray,
    edi_detrended: np.ndarray,
    edi_smooth: np.ndarray,
    merged_peak_mask: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if len(labels) == 0:
        return pd.DataFrame(), pd.DataFrame()

    segment_rows = []
    segment_id_per_sample = np.empty(len(labels), dtype=object)
    segment_label_per_sample = np.empty(len(labels), dtype=object)

    type_counts = {"breath": 0, "apnea": 0, "not_known": 0}
    seg_idx = 0
    start = 0

    for i in range(1, len(labels) + 1):
        boundary = i == len(labels) or labels[i] != labels[start]
        if not boundary:
            continue

        seg_type = str(labels[start])
        type_counts[seg_type] += 1
        segment_id = f"SEG_{seg_idx:06d}"
        segment_label = f"{seg_type}_{type_counts[seg_type]:04d}"
        end = i - 1

        linked_ids = breath_id_per_sample.iloc[start : end + 1].dropna().astype(int).unique().tolist()

        segment_rows.append(
            {
                "patient_id": patient_id,
                "segment_id": segment_id,
                "segment_label": segment_label,
                "segment_type": seg_type,
                "start_idx": int(start),
                "end_idx": int(end),
                "start_time": float(ts[start]),
                "end_time": float(ts[end]),
                "start_time_sec_from_start": float(time_sec[start]),
                "end_time_sec_from_start": float(time_sec[end]),
                "duration_sec": float(time_sec[end] - time_sec[start]),
                "n_samples": int(end - start + 1),
                "n_peaks": int(merged_peak_mask[start : end + 1].sum()),
                "linked_breath_ids": linked_ids,
                "edi_signal_raw": edi_raw[start : end + 1].astype(float).tolist(),
                "edi_signal_detrended": edi_detrended[start : end + 1].astype(float).tolist(),
                "edi_signal_smooth": edi_smooth[start : end + 1].astype(float).tolist(),
            }
        )

        segment_id_per_sample[start : end + 1] = segment_id
        segment_label_per_sample[start : end + 1] = segment_label
        seg_idx += 1
        start = i

    sample_segment_df = pd.DataFrame(
        {
            "segment_id": segment_id_per_sample,
            "segment_label": segment_label_per_sample,
            "segment_type": labels,
        }
    )
    return pd.DataFrame(segment_rows), sample_segment_df


def iter_patient_files(folder: Path) -> Iterable[Path]:
    return sorted(folder.glob("movingwinddetected_patient_*.xlsx"))


def run_one_patient(xlsx_path: Path, out_dir: Path) -> PatientRunResult:
    patient_id = get_patient_id_from_path(xlsx_path)
    print(f"[START] patient {patient_id} | {xlsx_path.name}")

    try:
        df = pd.read_excel(xlsx_path)
        required = [TIMESTAMP_COL, EDI_COL] + PEAK_COLS
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise KeyError(f"Missing required columns: {missing}")

        ts = pd.to_numeric(df[TIMESTAMP_COL], errors="coerce").to_numpy()
        edi_raw = pd.to_numeric(df[EDI_COL], errors="coerce").to_numpy()
        peak_masks_raw = {c: to_bool_series(df[c]).to_numpy() for c in PEAK_COLS}

        valid = np.isfinite(ts) & np.isfinite(edi_raw)
        if valid.sum() < 10:
            raise ValueError("Too few valid samples.")

        df = df.loc[valid].reset_index(drop=True)
        ts = ts[valid]
        edi_raw = edi_raw[valid]
        peak_masks_raw = {k: v[valid] for k, v in peak_masks_raw.items()}

        fs = infer_sample_rate_hz(ts, SAMPLE_RATE_HZ)
        is_ms = np.median(np.diff(ts)) > 1.5
        time_sec = (ts - ts[0]) / (1000.0 if is_ms else 1.0)

        baseline_win = max(3, int(round(BASELINE_MEDIAN_SEC * fs)))
        smooth_win = max(3, int(round(SMOOTHING_SEC * fs)))
        baseline = rolling_median_np(edi_raw, baseline_win)
        edi_detrended = edi_raw - baseline
        edi_smooth = rolling_mean_np(edi_detrended, smooth_win)

        apnea_mask, apnea_cols = build_apnea_mask(df)
        analysis_window_mask = (time_sec >= EDGE_EXCLUDE_SEC) & (time_sec <= (time_sec[-1] - EDGE_EXCLUDE_SEC))

        cand_mask = np.zeros(len(df), dtype=bool)
        for c in PEAK_COLS:
            cand_mask |= peak_masks_raw[c]
        cand_mask &= (~apnea_mask) & analysis_window_mask
        candidate_idx = np.flatnonzero(cand_mask)
        peak_idx = merge_close_peaks(candidate_idx, time_sec, edi_smooth)

        breaths_df = build_breaths_all(
            ts=ts,
            edi_raw=edi_raw,
            edi_detrended=edi_detrended,
            edi_smooth=edi_smooth,
            peak_idx=peak_idx,
            analysis_window_mask=analysis_window_mask,
            patient_id=patient_id,
            is_ms=is_ms,
        )

        labels, breath_id_per_sample = assign_sample_labels(len(df), apnea_mask, breaths_df)
        merged_peak_mask = np.zeros(len(df), dtype=bool)
        merged_peak_mask[peak_idx] = True
        segment_df, sample_segment_df = build_segment_catalog(
            patient_id=patient_id,
            ts=ts,
            time_sec=time_sec,
            labels=labels,
            breath_id_per_sample=breath_id_per_sample,
            edi_raw=edi_raw,
            edi_detrended=edi_detrended,
            edi_smooth=edi_smooth,
            merged_peak_mask=merged_peak_mask,
        )

        patient_out = out_dir / f"patient_{patient_id}"
        patient_out.mkdir(parents=True, exist_ok=True)

        sample_level_df = pd.DataFrame(
            {
                "patient_id": patient_id,
                "sample_idx": np.arange(len(df), dtype=int),
                "timestamp": ts.astype(float),
                "time_sec_from_start": time_sec.astype(float),
                "edi_raw": edi_raw.astype(float),
                "edi_baseline_median": baseline.astype(float),
                "edi_detrended": edi_detrended.astype(float),
                "edi_smooth_for_detection": edi_smooth.astype(float),
                "analysis_window_mask": analysis_window_mask.astype(bool),
                "apnea_mask": apnea_mask.astype(bool),
                "candidate_peak_mask": cand_mask.astype(bool),
                "merged_peak_mask": merged_peak_mask.astype(bool),
                "breath_id": breath_id_per_sample.astype("Int64"),
                "is_breath": labels == "breath",
                "is_apnea": labels == "apnea",
            }
        )
        sample_level_df = pd.concat([sample_level_df, sample_segment_df], axis=1)

        sample_level_df.to_pickle(patient_out / f"patient_{patient_id}_sample_level_signal.pkl")
        sample_level_df.to_csv(patient_out / f"patient_{patient_id}_sample_level_signal.csv", index=False)
        segment_df.to_pickle(patient_out / f"patient_{patient_id}_segment_catalog.pkl")
        segment_df.to_csv(patient_out / f"patient_{patient_id}_segment_catalog.csv", index=False)
        breaths_df.to_pickle(patient_out / f"patient_{patient_id}_detected_breaths.pkl")
        breaths_df.to_csv(patient_out / f"patient_{patient_id}_detected_breaths.csv", index=False)

        run_meta = pd.DataFrame(
            [
                {
                    "patient_id": patient_id,
                    "n_rows": int(len(df)),
                    "n_apnea_label_columns": int(len(apnea_cols)),
                    "n_candidate_peaks": int(len(candidate_idx)),
                    "n_peaks_merged": int(len(peak_idx)),
                    "n_detected_breaths": int(len(breaths_df)),
                    "n_segments_total": int(len(segment_df)),
                    "n_segments_breath": int((segment_df["segment_type"] == "breath").sum()),
                    "n_segments_apnea": int((segment_df["segment_type"] == "apnea").sum()),
                    "n_segments_not_known": int((segment_df["segment_type"] == "not_known").sum()),
                }
            ]
        )
        run_meta.to_csv(patient_out / f"patient_{patient_id}_run_meta.csv", index=False)

        print(
            f"[DONE] patient {patient_id} | breaths={len(breaths_df)} | segments={len(segment_df)}"
        )
        return PatientRunResult(
            patient_id=patient_id,
            n_rows=int(len(df)),
            n_breath_segments=int((segment_df["segment_type"] == "breath").sum()),
            n_apnea_segments=int((segment_df["segment_type"] == "apnea").sum()),
            n_not_known_segments=int((segment_df["segment_type"] == "not_known").sum()),
            n_total_segments=int(len(segment_df)),
            status="ok",
            message="",
        )

    except Exception as e:  # noqa: BLE001
        print(f"[ERROR] patient {patient_id} | {e}")
        return PatientRunResult(patient_id, 0, 0, 0, 0, 0, "error", str(e))


def main() -> None:
    in_dir = PARENT_DIR / ANALYSIS_FOLDER
    if not in_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {in_dir}")

    out_dir = OUT_ROOT / ANALYSIS_FOLDER
    out_dir.mkdir(parents=True, exist_ok=True)

    patient_files = list(iter_patient_files(in_dir))
    if not patient_files:
        raise FileNotFoundError(f"No patient files found in {in_dir}")

    results = []
    segment_catalogs = []
    for fp in patient_files:
        res = run_one_patient(fp, out_dir)
        results.append(res)
        if res.status != "ok":
            continue
        pid = res.patient_id
        seg_fp = out_dir / f"patient_{pid}" / f"patient_{pid}_segment_catalog.csv"
        if seg_fp.exists():
            seg_df = pd.read_csv(seg_fp)
            segment_catalogs.append(seg_df)

    run_summary_df = pd.DataFrame([r.__dict__ for r in results])
    run_summary_df.to_csv(out_dir / "all_patients_run_summary.csv", index=False)

    if segment_catalogs:
        all_segments_df = pd.concat(segment_catalogs, ignore_index=True)
        all_segments_df.to_csv(out_dir / "all_patients_segment_catalog.csv", index=False)

    print("\n=== Completed ===")
    print(f"Input:  {in_dir}")
    print(f"Output: {out_dir}")
    print("Saved patient-wise sample-level signals and segment catalogs.")


In [ ]:
main()
